# Robot repartidor de galletas con Programación Genética (PG)

Un robot recorre una sala cuadrada llena de ingenieros de diseño de robots y entrega galletas. Cada ingeniero que recibe su primera galleta da puntos. La PG evoluciona **el programa (árbol)** que controla al robot.

**Parámetros inventados (modificables en la celda 2):**
- Sala: cuadrícula 8×8, 12 ingenieros en posiciones fijas (semilla).
- El robot inicia en la esquina (0,0) mirando al este.
- Máximo 200 pasos y 20 galletas.
- 10 puntos por ingeniero atendido.

**Elementos de la PG:**
| Elemento | Definición |
|---|---|
| Terminales | `AVANZAR`, `GIRAR_IZQ`, `GIRAR_DER`, `ENTREGAR` |
| Funciones | `IF_ING_ADELANTE(A,B)`, `IF_PARED_ADELANTE(A,B)`, `PROGN2(A,B)`, `PROGN3(A,B,C)` |
| Aptitud | puntos − α·pasos − β·entregas inválidas − γ·galletas repetidas − λ·tamaño |

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# ---------------- Parámetros del problema ----------------
N = 8                 # tamaño de la sala (N x N)
N_ING = 12            # número de ingenieros
T = 200               # máximo de pasos (acciones) del robot
GALLETAS = 20         # galletas disponibles
PUNTOS = 10           # puntos por ingeniero atendido

# Pesos de la función de aptitud
ALPHA = 0.02          # penalización por paso usado
BETA = 0.5            # penalización por entregar donde no hay ingeniero
GAMMA = 1.0           # penalización por darle otra galleta a quien ya tiene
LAMBDA = 0.01         # penalización por tamaño del árbol (parsimonia)

# ---------------- Parámetros de la PG ----------------
POP = 300
GENS = 40
TORNEO = 5
P_CRUCE = 0.85
P_MUT = 0.10
MAX_PROF_INI = 5
MAX_PROF = 8
ELITE = 2
SEMILLA = 1

# ---------------- Sala ----------------
rng_sala = random.Random(7)
celdas = [(r, c) for r in range(N) for c in range(N) if (r, c) != (0, 0)]
INGENIEROS = frozenset(rng_sala.sample(celdas, N_ING))
print("Ingenieros en:", sorted(INGENIEROS))

## 1. Conjuntos de terminales y funciones
Los árboles se representan como listas anidadas: `['PROGN2', A, B]`; los terminales son cadenas.

In [ ]:
TERMINALES = ['AVANZAR', 'GIRAR_IZQ', 'GIRAR_DER', 'ENTREGAR']
FUNCIONES = {            # nombre: aridad
    'IF_ING_ADELANTE': 2,   # si hay ingeniero sin galleta enfrente -> A, si no -> B
    'IF_PARED_ADELANTE': 2, # si hay pared enfrente -> A, si no -> B
    'PROGN2': 2,            # ejecuta A y luego B
    'PROGN3': 3,            # ejecuta A, B y C
}
DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]   # Este, Sur, Oeste, Norte

## 2. Simulador del robot (intérprete del árbol)
Cada terminal consume un paso. El programa se repite en bucle hasta que se acaban los pasos o las galletas, o hasta que todos los ingenieros están atendidos.

`ENTREGAR` da una galleta al ingeniero que está en la **celda actual** del robot.

In [ ]:
class Simulador:
    def __init__(self, grabar=False):
        self.grabar = grabar      # si True guarda cada paso para poder animarlo
        self.frames = []
        self.pos = (0, 0)
        self.dir = 0
        self.pasos = 0
        self.galletas = GALLETAS
        self.atendidos = set()
        self.invalidas = 0
        self.repetidas = 0
        self.traza = [self.pos]
        self._registrar(None)

    def puntaje(self):
        return (PUNTOS * len(self.atendidos) - BETA * self.invalidas
                - GAMMA * self.repetidas)

    def _registrar(self, evento):
        if self.grabar:
            self.frames.append(dict(pos=self.pos, dir=self.dir, pasos=self.pasos,
                                    galletas=self.galletas, score=self.puntaje(),
                                    atendidos=frozenset(self.atendidos), evento=evento))

    def terminado(self):
        return (self.pasos >= T or self.galletas <= 0
                or len(self.atendidos) == len(INGENIEROS))

    def adelante(self):
        dr, dc = DIRS[self.dir]
        return (self.pos[0] + dr, self.pos[1] + dc)

    def hay_pared(self):
        r, c = self.adelante()
        return not (0 <= r < N and 0 <= c < N)

    def hay_ing_adelante(self):
        return self.adelante() in INGENIEROS and self.adelante() not in self.atendidos

    def accion(self, a):
        if self.terminado():
            return
        self.pasos += 1
        if a == 'AVANZAR':
            if not self.hay_pared():
                self.pos = self.adelante()
        elif a == 'GIRAR_IZQ':
            self.dir = (self.dir - 1) % 4
        elif a == 'GIRAR_DER':
            self.dir = (self.dir + 1) % 4
        evento = None
        if a == 'ENTREGAR':
            self.galletas -= 1
            if self.pos in INGENIEROS:
                if self.pos in self.atendidos:
                    self.repetidas += 1
                    evento = 'repetida'
                else:
                    self.atendidos.add(self.pos)
                    evento = 'ok'
            else:
                self.invalidas += 1
                evento = 'invalida'
        self.traza.append(self.pos)
        self._registrar(evento)

    def ejecutar(self, nodo):
        if self.terminado():
            return
        if isinstance(nodo, str):
            self.accion(nodo)
            return
        f = nodo[0]
        if f == 'IF_ING_ADELANTE':
            self.ejecutar(nodo[1] if self.hay_ing_adelante() else nodo[2])
        elif f == 'IF_PARED_ADELANTE':
            self.ejecutar(nodo[1] if self.hay_pared() else nodo[2])
        else:  # PROGN2 / PROGN3
            for hijo in nodo[1:]:
                self.ejecutar(hijo)

    def correr(self, arbol):
        while not self.terminado():
            self.ejecutar(arbol)   # cada camino del árbol ejecuta >= 1 acción
        return self

## 3. Función de aptitud
$$F = \text{PUNTOS}\cdot |\text{atendidos}| - \alpha\,\text{pasos} - \beta\,\text{inválidas} - \gamma\,\text{repetidas} - \lambda\,|\text{árbol}|$$

Aptitud máxima teórica: $12 \cdot 10 = 120$ (menos las penalizaciones por pasos).

In [ ]:
def tamano(t):
    return 1 if isinstance(t, str) else 1 + sum(tamano(h) for h in t[1:])

def profundidad(t):
    return 0 if isinstance(t, str) else 1 + max(profundidad(h) for h in t[1:])

def aptitud(arbol):
    s = Simulador().correr(arbol)
    return (PUNTOS * len(s.atendidos) - ALPHA * s.pasos - BETA * s.invalidas
            - GAMMA * s.repetidas - LAMBDA * tamano(arbol))

F_MAX = PUNTOS * N_ING

## 4. Generación de árboles y operadores genéticos

In [ ]:
def generar(prof, metodo):
    # metodo: 'full' o 'grow'
    if prof == 0 or (metodo == 'grow' and random.random() < 0.3):
        return random.choice(TERMINALES)
    f = random.choice(list(FUNCIONES))
    return [f] + [generar(prof - 1, metodo) for _ in range(FUNCIONES[f])]

def poblacion_inicial():
    # Ramped half-and-half
    pob = []
    for i in range(POP):
        prof = 2 + i % (MAX_PROF_INI - 1)
        pob.append(generar(prof, 'full' if i % 2 else 'grow'))
    return pob

def nodos(t, ruta=()):
    yield ruta, t
    if not isinstance(t, str):
        for i, h in enumerate(t[1:], 1):
            yield from nodos(h, ruta + (i,))

def reemplazar(t, ruta, nuevo):
    if not ruta:
        return nuevo
    copia = list(t)
    copia[ruta[0]] = reemplazar(t[ruta[0]], ruta[1:], nuevo)
    return copia

def cruce(a, b):
    ra, sa = random.choice(list(nodos(a)))
    rb, sb = random.choice(list(nodos(b)))
    h1, h2 = reemplazar(a, ra, sb), reemplazar(b, rb, sa)
    if profundidad(h1) > MAX_PROF: h1 = a
    if profundidad(h2) > MAX_PROF: h2 = b
    return h1, h2

def mutar(a):
    ruta, _ = random.choice(list(nodos(a)))
    h = reemplazar(a, ruta, generar(3, 'grow'))
    return h if profundidad(h) <= MAX_PROF else a

def torneo(pob, fit):
    ids = random.sample(range(len(pob)), TORNEO)
    return pob[max(ids, key=lambda i: fit[i])]

## 5. Bucle evolutivo

In [ ]:
def evolucionar():
    random.seed(SEMILLA)
    pob = poblacion_inicial()
    hist_max, hist_prom = [], []
    mejor, mejor_f = None, -1e9
    for g in range(GENS + 1):
        fit = [aptitud(a) for a in pob]
        i_best = max(range(POP), key=lambda i: fit[i])
        if fit[i_best] > mejor_f:
            mejor, mejor_f = pob[i_best], fit[i_best]
        hist_max.append(mejor_f)
        hist_prom.append(sum(fit) / POP)
        if g % 5 == 0 or g == GENS:
            print(f"Gen {g:3d} | mejor = {mejor_f:7.2f} | promedio = {hist_prom[-1]:7.2f}")
        if len(Simulador().correr(mejor).atendidos) == N_ING and g > 0 and mejor_f > F_MAX - 10:
            print("Solución casi óptima encontrada, se detiene.")
            break
        # nueva generación con elitismo
        orden = sorted(range(POP), key=lambda i: fit[i], reverse=True)
        nueva = [pob[i] for i in orden[:ELITE]]
        while len(nueva) < POP:
            p1, p2 = torneo(pob, fit), torneo(pob, fit)
            if random.random() < P_CRUCE:
                h1, h2 = cruce(p1, p2)
            else:
                h1, h2 = p1, p2
            if random.random() < P_MUT: h1 = mutar(h1)
            if random.random() < P_MUT: h2 = mutar(h2)
            nueva += [h1, h2]
        pob = nueva[:POP]
    return mejor, mejor_f, hist_max, hist_prom

mejor, mejor_f, hist_max, hist_prom = evolucionar()

## 6. Resultados

In [ ]:
def a_texto(t, nivel=0):
    if isinstance(t, str):
        return "  " * nivel + t
    s = "  " * nivel + t[0] + "(\n"
    s += ",\n".join(a_texto(h, nivel + 1) for h in t[1:])
    return s + "\n" + "  " * nivel + ")"

s = Simulador(grabar=True).correr(mejor)
print("Mejor programa encontrado:\n")
print(a_texto(mejor))
print(f"\nAptitud: {mejor_f:.2f} (máx. teórica {F_MAX})")
print(f"Ingenieros atendidos: {len(s.atendidos)}/{N_ING}")
print(f"Pasos usados: {s.pasos}/{T} | Galletas usadas: {GALLETAS - s.galletas}/{GALLETAS}")
print(f"Entregas inválidas: {s.invalidas} | Galletas repetidas: {s.repetidas}")
print(f"Tamaño del árbol: {tamano(mejor)} nodos, profundidad {profundidad(mejor)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Curva de evolución
ax[0].plot(hist_max, label='Mejor')
ax[0].plot(hist_prom, label='Promedio')
ax[0].axhline(F_MAX, color='gray', ls='--', label='Máximo teórico')
ax[0].set_xlabel('Generación'); ax[0].set_ylabel('Aptitud')
ax[0].set_title('Evolución de la aptitud'); ax[0].legend()

# Sala y recorrido
ax[1].set_xlim(-0.5, N - 0.5); ax[1].set_ylim(N - 0.5, -0.5)
ax[1].set_xticks(range(N)); ax[1].set_yticks(range(N)); ax[1].grid(True, alpha=0.3)
for (r, c) in INGENIEROS:
    atendido = (r, c) in s.atendidos
    ax[1].scatter(c, r, s=250, marker='s', color='tab:green' if atendido else 'tab:red', zorder=3)
xs = [p[1] for p in s.traza]; ys = [p[0] for p in s.traza]
ax[1].plot(xs, ys, '-', color='tab:blue', alpha=0.6, zorder=2)
ax[1].scatter(0, 0, s=120, marker='o', color='black', zorder=4, label='Inicio')
ax[1].scatter(xs[-1], ys[-1], s=120, marker='*', color='gold', edgecolor='k', zorder=4, label='Fin')
ax[1].set_title('Recorrido (verde = con galleta, rojo = sin galleta)')
ax[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

## 7. Ver al robot repartiendo las galletas (estilo juego)
Igual que en el Pong del notebook de aprendizaje por refuerzo: en cada paso se limpia la salida (`clear_output`), se dibuja el cuadro con `matplotlib` y se hace una pausa.

- Robot dorado (el triángulo indica hacia dónde mira)
- Ingeniero rojo = sin galleta, verde = ya atendido
- Al entregar aparece la galleta 🍪 sobre el ingeniero; si se entrega donde no hay nadie aparece una ✗

In [ ]:
from time import sleep
from IPython.display import clear_output

VEC = [(1, 0), (0, 1), (-1, 0), (0, -1)]   # Este, Sur, Oeste, Norte en (x, y)

def dibujar_frame(f, final=False):
    fig, ax = plt.subplots(figsize=(6, 6.4))
    ax.set_xlim(-0.5, N - 0.5); ax.set_ylim(N - 0.5, -1.4)
    ax.set_xticks(range(N)); ax.set_yticks(range(N)); ax.set_aspect('equal')
    ax.set_xticks([i - 0.5 for i in range(N + 1)], minor=True)
    ax.set_yticks([i - 0.5 for i in range(N + 1)], minor=True)
    ax.grid(which='minor', color='lightgray'); ax.tick_params(which='both', length=0)

    # Ingenieros
    for (r, c) in INGENIEROS:
        ok = (r, c) in f['atendidos']
        ax.add_patch(plt.Rectangle((c - 0.35, r - 0.35), 0.7, 0.7,
                                   fc='mediumseagreen' if ok else 'indianred', ec='black', zorder=2))
        ax.add_patch(plt.Circle((c, r - 0.08), 0.17, fc='bisque', ec='black', zorder=3))  # cabeza
        ax.add_patch(plt.Rectangle((c - 0.22, r + 0.12), 0.44, 0.15, fc='navy', ec='black', zorder=3))  # cuerpo
        if ok:
            ax.add_patch(plt.Circle((c + 0.28, r + 0.28), 0.12, fc='peru', ec='saddlebrown', zorder=3))

    # Robot (triángulo que apunta en su dirección)
    r, c = f['pos']
    vx, vy = VEC[f['dir']]
    px, py = -vy, vx
    tri = [(c + 0.42 * vx, r + 0.42 * vy),
           (c - 0.3 * vx + 0.3 * px, r - 0.3 * vy + 0.3 * py),
           (c - 0.3 * vx - 0.3 * px, r - 0.3 * vy - 0.3 * py)]
    ax.add_patch(plt.Polygon(tri, fc='gold', ec='black', zorder=5))

    # Evento de entrega
    if f['evento'] in ('ok', 'repetida'):
        ax.add_patch(plt.Circle((c, r - 0.62), 0.2, fc='peru', ec='saddlebrown', lw=2, zorder=6))
        ax.text(c, r - 0.62, '•', ha='center', va='center', color='saddlebrown', fontsize=12, zorder=7)
        ax.text(c, r - 0.95, '+10' if f['evento'] == 'ok' else 'repetida',
                ha='center', color='darkgreen' if f['evento'] == 'ok' else 'darkorange',
                fontsize=11, weight='bold', zorder=7)
    elif f['evento'] == 'invalida':
        ax.text(c, r - 0.7, '✗', ha='center', color='red', fontsize=18, weight='bold', zorder=7)

    ax.text(-0.4, -0.95, f"SCORE: {f['score']:.1f}   GALLETAS: {f['galletas']}   PASOS: {f['pasos']}/{T}",
            fontsize=12, va='center')
    if final:
        if len(f['atendidos']) == N_ING:
            ax.text(N / 2 - 0.5, N / 2 - 0.5, 'YOU WIN!', fontsize=30, color='darkgreen',
                    ha='center', va='center', weight='bold', zorder=10,
                    bbox=dict(fc='white', ec='darkgreen', alpha=0.85))
        else:
            ax.text(N / 2 - 0.5, N / 2 - 0.5, 'GAME OVER', fontsize=30, color='darkred',
                    ha='center', va='center', weight='bold', zorder=10,
                    bbox=dict(fc='white', ec='darkred', alpha=0.85))
    return fig

def animar(arbol, pausa=0.15):
    juego = Simulador(grabar=True).correr(arbol)
    for i, f in enumerate(juego.frames):
        clear_output(wait=True)
        fig = dibujar_frame(f, final=(i == len(juego.frames) - 1))
        plt.show()
        plt.close(fig)
        sleep(pausa)
    return juego

# Ejecuta esta celda para ver jugar al mejor robot (pausa más pequeña = más rápido)
juego = animar(mejor, pausa=0.15)

**Tip:** si la animación te parece muy lenta o rápida, cambia `pausa`. Si quieres repetirla, vuelve a ejecutar `animar(mejor)`.

## 8. Análisis rápido
- La PG no recibe el recorrido: descubre por sí sola una combinación de giros, avances y entregas que maximiza los puntos.
- Los **sensores** (`IF_ING_ADELANTE`, `IF_PARED_ADELANTE`) permiten comportamientos reactivos y evitan que el robot se quede pegado a la pared.
- Los términos de penalización (pasos, entregas inválidas, galletas repetidas, tamaño) empujan a soluciones eficientes y sin *bloat*.
- Para experimentar: cambia `N`, `N_ING`, `T`, los pesos o `SEMILLA` en la celda 2 y vuelve a ejecutar todo.